In [ ]:
%pip install sentence_transformers

In [ ]:
%pip install bertopic umap-learn

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from gensim.models import TfidfModel
from gensim.corpora import Dictionary, MmCorpus
from sklearn.cluster import KMeans

d:\TB2\Intro to AI and TEXT Analytics (EMATM0067)\Coursework\TextAnalytics-CW-Task4\venv3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csv_file = Path("../data/customer_support_tickets_preprocessed.csv")
df = pd.read_csv(csv_file)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Ticket Description charlength,Ticket Description wordlength,Tfidf_ticket_description,Embeddings_ticket_description
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,290,43,billing code appreciate requested website addr...,i'm having an issue with the . please assist. ...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,288,44,need change existing product facing intermitte...,i'm having an issue with the . please assist. ...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,277,42,facing turning working fine yesterday respond ...,i'm facing a problem with my . the is not turn...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,264,41,interested love happen check feedback contacte...,i'm having an issue with the . please assist. ...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,336,55,note seller responsible damage arising deliver...,i'm having an issue with the . please assist. ...


### TF-IDF - sklearn

In [3]:
pd.set_option('display.max_colwidth', None)

In [4]:
tfidf_vectorizer = TfidfVectorizer(max_df = 0.90, min_df = 5, ngram_range = (1,2))
X_tfidf = tfidf_vectorizer.fit_transform(df['Tfidf_ticket_description'])

In [5]:
X_tfidf.shape

(8067, 2127)

In [7]:
joblib.dump( X_tfidf, "../data/tfidf_sklearn.pkl")

['../data/tfidf_sklearn.pkl']

### TF-IDF - genism

In [8]:
dictionary = Dictionary.load("../data/gensim_lda_dictionary.dict")
bow_corpus = MmCorpus("../data/gensim_lda_bow_corpus.mm")

In [9]:
tfidf_model = TfidfModel(bow_corpus)
tfidf_corpus = [tfidf_model[doc] for doc in bow_corpus]
tfidf_corpus

[[(0, np.float64(0.408964195161081)),
  (1, np.float64(0.34739064916880585)),
  (2, np.float64(0.34164490553090027)),
  (3, np.float64(0.2250661781231973)),
  (4, np.float64(0.23151494307617454)),
  (5, np.float64(0.4533586794159697)),
  (6, np.float64(0.18840681094401288)),
  (7, np.float64(0.16817711386114698)),
  (8, np.float64(0.166990652095419)),
  (9, np.float64(0.2966290099608657)),
  (10, np.float64(0.10271480266880795)),
  (11, np.float64(0.12214235080931823)),
  (12, np.float64(0.14302140418702464)),
  (13, np.float64(0.15459913668028863)),
  (14, np.float64(0.18994294334340422))],
 [(15, np.float64(0.2928351084694365)),
  (16, np.float64(0.2715491823608704)),
  (17, np.float64(0.632644512053193)),
  (18, np.float64(0.23985772505106948)),
  (19, np.float64(0.2336750040721554)),
  (20, np.float64(0.2928351084694365)),
  (21, np.float64(0.21875646957148723)),
  (22, np.float64(0.15601348488030647)),
  (23, np.float64(0.20340578822512242)),
  (24, np.float64(0.2928351084694365))

In [10]:
MmCorpus.serialize("../data/gensim_lda_tfidf_corpus.mm", tfidf_corpus)

### Sentence Embeddings

In [11]:
sentence_embedding = SentenceTransformer('all-MiniLM-L6-v2')
X_embed = sentence_embedding.encode(df['Embeddings_ticket_description'].tolist(), show_progress_bar = True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 760.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 253/253 [09:02<00:00,  2.14s/it]


In [12]:
X_embed.shape

(8067, 384)

In [13]:
np.save('../data/sentence_embeddings.npy', X_embed)